# Experimenting with using DBSCAN instead of KDE to create the outbreak cores

In [2]:
import numpy as np
import pandas as pd

In [4]:
## Load in original report data:
reports_tagged = pd.read_csv('data/reports_tagged_initial.csv', low_memory=False)

In [5]:
reports_tagged

,EVENT_ID,time_hour,report_lat,report_lon,EVENT_TYPE,BEGIN_DT_UTC,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,...,dist_km,rlon,rlat,track_start,track_end,time_min_msl,min_msl,n_steps,lifecycle_frac,hours_from_min_msl
0,10351815,1995-12-17 16:00:00+00:00,29.4700,-95.0500,Tornado,1995-12-17 15:45:00+00:00,199512,17,945,199512,...,565.361833,4.2000,3.4700,1995-12-16 21:00:00+00:00,1995-12-26 13:00:00+00:00,1995-12-21 18:00:00+00:00,970.2875,212.0,0.081897,-98.0
1,10328563,1995-12-17 20:00:00+00:00,31.2200,-92.1700,Tornado,1995-12-17 19:40:00+00:00,199512,17,1340,199512,...,942.067203,6.8300,5.9700,1995-12-16 21:00:00+00:00,1995-12-26 13:00:00+00:00,1995-12-21 18:00:00+00:00,970.2875,212.0,0.099138,-94.0
2,10327292,1995-12-17 20:00:00+00:00,30.5500,-91.5500,Tornado,1995-12-17 20:12:00+00:00,199512,17,1412,199512,...,939.379509,7.4500,5.3000,1995-12-16 21:00:00+00:00,1995-12-26 13:00:00+00:00,1995-12-21 18:00:00+00:00,970.2875,212.0,0.099138,-94.0
3,10327420,1995-12-18 19:00:00+00:00,30.6000,-90.2300,Tornado,1995-12-18 19:15:00+00:00,199512,18,1315,199512,...,386.951821,2.2700,-2.9000,1995-12-16 21:00:00+00:00,1995-12-26 13:00:00+00:00,1995-12-21 18:00:00+00:00,970.2875,212.0,0.198276,-71.0
4,10354458,1995-12-17 14:00:00+00:00,28.8700,-96.2200,Tornado,1995-12-17 14:10:00+00:00,199512,17,810,199512,...,404.309660,3.5300,1.8700,1995-12-16 21:00:00+00:00,1995-12-26 13:00:00+00:00,1995-12-21 18:00:00+00:00,970.2875,212.0,0.073276,-100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83964,1290825,2025-03-31 03:00:00+00:00,38.1571,-83.3841,Thunderstorm Wind,2025-03-31 03:20:00+00:00,202503,30,2320,202503,...,709.138149,-0.8841,-6.3429,2025-03-26 22:00:00+00:00,2025-03-31 23:00:00+00:00,2025-03-31 20:00:00+00:00,992.6212,114.0,0.834711,-17.0
83965,1290828,2025-03-31 04:00:00+00:00,37.1871,-83.7779,Thunderstorm Wind,2025-03-31 04:20:00+00:00,202503,31,20,202503,...,847.740553,-1.2779,-7.5629,2025-03-26 22:00:00+00:00,2025-03-31 23:00:00+00:00,2025-03-31 20:00:00+00:00,992.6212,114.0,0.842975,-16.0
83966,1290851,2025-03-31 05:00:00+00:00,37.3389,-82.5863,Thunderstorm Wind,2025-03-31 05:19:00+00:00,202503,31,119,202503,...,892.881198,-1.8363,-7.9111,2025-03-26 22:00:00+00:00,2025-03-31 23:00:00+00:00,2025-03-31 20:00:00+00:00,992.6212,114.0,0.851240,-15.0
83967,1249986,2025-03-27 07:00:00+00:00,26.0711,-97.1590,Thunderstorm Wind,2025-03-27 07:00:00+00:00,202503,27,200,202503,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Stage A: core detection: